# SimCLR Contrastive Training for UCI-HAR Dataset (6 channels: acc + gyro)

In [ ]:
# Author: C. I. Tang
# Adapted for UCI-HAR (6 channels) by Mochi
# Based on work of Tang et al.: https://arxiv.org/abs/2011.11542
# License: GNU General Public License v3.0

%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
import pickle
import scipy
import datetime
import numpy as np
import tensorflow as tf

seed = 1
tf.random.set_seed(seed)
np.random.seed(seed)

In [ ]:
# Libraries for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.manifold

sns.set_context('poster')

In [ ]:
# Library scripts
import raw_data_processing
import data_pre_processing
import simclr_models
import simclr_utitlities
import transformations

In [ ]:
working_directory = 'test_run/'
dataset_save_path = working_directory
if not os.path.exists(working_directory):
    os.mkdir(working_directory)

## UCI-HAR Dataset (6 channels)

UCI Human Activity Recognition Using Smartphones Dataset
Uses 6 channels: body_acc + body_gyro (xyz each)

In [ ]:
# Load UCI-HAR with 6 channels (acc + gyro)
data_folder_path = 'datasets/UCI-HAR/'
user_datasets = raw_data_processing.process_uci_har_6channels(data_folder_path)

## Pre-processing

In [ ]:
# Parameters
# UCI-HAR: 50Hz sampling rate, each window is already 128 timesteps
window_size = 128
input_shape = (window_size, 6)  # 6 channels: acc_xyz + gyro_xyz

# Dataset Metadata 
transformation_multiple = 1
dataset_name = 'ucihar_6ch.pkl'
dataset_name_user_split = 'ucihar_6ch_user_split.pkl'

# UCI-HAR labels
label_list = [
    'null', 
    'WALKING', 
    'WALKING_UPSTAIRS', 
    'WALKING_DOWNSTAIRS', 
    'SITTING', 
    'STANDING', 
    'LAYING'
]
label_map = dict([(l, i) for i, l in enumerate(label_list)])
output_shape = len(label_list)

model_save_name = f"ucihar_6ch_acc"

# Original UCI-HAR already has train/test split, we'll keep that
def get_fixed_split_users(har_users):
    # UCI-HAR original split: train = subjects 1-30 except test, test =  2, 4, 9, 10, 12, 13, 18, 20, 24
    test_users = [2, 4, 9, 10, 12, 13, 18, 20, 24]
    train_users = [u for u in har_users if u not in test_users]
    return (train_users, test_users)

In [ ]:
with open(dataset_save_path + dataset_name_user_split, 'wb') as f:
    pickle.dump({
        'user_split': user_datasets,
    }, f)

In [ ]:
har_users = list(user_datasets.keys())
train_users, test_users = get_fixed_split_users(har_users)
print(f'Testing: {test_users}, Training: {train_users}')

In [ ]:
np_train, np_val, np_test = data_pre_processing.pre_process_dataset_composite(
    user_datasets=user_datasets, 
    label_map=label_map, 
    output_shape=output_shape, 
    train_users=train_users, 
    test_users=test_users, 
    window_size=window_size, 
    shift=window_size//2, 
    normalise_dataset=True, 
    verbose=1
)


## SimCLR Training

In [ ]:
batch_size = 512
decay_steps = 1000
epochs = 400
temperature = 0.05
transform_funcs = [
    transformations.noise_transform_vectorized, # Add Gaussian noise
    transformations.scaling_transform_vectorized, # Random scaling
    transformations.rotation_transform_vectorized # Random 3D rotation
]
# 🔍 DEBUG: Test 6-channel transformation functions before training
import numpy as np

print("🔍 Testing rotation_transform_vectorized with 6-channel input...")
test_6ch = np.random.randn(2, 128, 6)  # (batch, time, channels)

try:
    result_6ch = transformations.rotation_transform_vectorized(test_6ch)
    
    print(f"✅ 输入形状: {test_6ch.shape}")
    print(f"✅ 输出形状: {result_6ch.shape}")
    print(f"✅ 形状一致: {test_6ch.shape == result_6ch.shape}")
    print(f"✅ 无 NaN: {not np.isnan(result_6ch).any()}")
    
    diff_acc = np.mean(np.abs(result_6ch[:, :, 0:3] - test_6ch[:, :, 0:3]))
    diff_gyro = np.mean(np.abs(result_6ch[:, :, 3:6] - test_6ch[:, :, 3:6]))
    print(f"✅ ACC 平均变化: {diff_acc:.4f}")
    print(f"✅ GYRO 平均变化: {diff_gyro:.4f}")
    
    assert 0.01 < diff_acc < 10.0, f"ACC 变化异常: {diff_acc}"
    assert 0.01 < diff_gyro < 10.0, f"GYRO 变化异常: {diff_gyro}"
    
    print("🎉 6-channel rotation test PASSED! 可以开始训练。")
    
except Exception as e:
    print(f"❌ 测试失败: {e}")
    print("💡 建议: 检查 transformations.py 中的 rotation_transform_vectorized 函数")
    raise  # 抛出错误，阻止后续训练

transformation_function = simclr_utitlities.generate_composite_transform_function_simple(transform_funcs)

In [ ]:
start_time = datetime.datetime.now()
start_time_str = start_time.strftime("%Y%m%d-%H%M%S")
tf.keras.backend.set_floatx('float32')

lr_decayed_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=0.01, decay_steps=decay_steps)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr_decayed_fn, clipnorm=1.0)

base_model = simclr_models.create_base_model(input_shape, model_name="base_model")
simclr_model = simclr_models.attach_simclr_head(base_model)
simclr_model.summary()

trained_simclr_model, epoch_losses = simclr_utitlities.simclr_train_model(simclr_model, np_train[0], optimizer, batch_size, transformation_function, temperature=temperature, epochs=epochs, is_trasnform_function_vectorized=True, verbose=1)

simclr_model_save_path = f"{working_directory}{start_time_str}_simclr.keras"
trained_simclr_model.save(simclr_model_save_path)

In [ ]:
plt.figure(figsize=(12,8))
plt.plot(epoch_losses)
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.show()

## Fine-tuning and Evaluation

### Linear Model

In [ ]:
total_epochs = 50
batch_size = 200

tag = "linear_eval"

simclr_model = tf.keras.models.load_model(simclr_model_save_path)
linear_evaluation_model = simclr_models.create_linear_model_from_base_model(simclr_model, output_shape, intermediate_layer=7)

linear_eval_best_model_file_name = f"{working_directory}{start_time_str}_simclr_{tag}.keras"
best_model_callback = tf.keras.callbacks.ModelCheckpoint(linear_eval_best_model_file_name,
    monitor='val_loss', mode='min', save_best_only=True, save_weights_only=False, verbose=0
)

# Compute class weights for balanced training
from sklearn.utils.class_weight import compute_class_weight
y_train = np.argmax(np_train[1], axis=1)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

training_history = linear_evaluation_model.fit(
    x = np_train[0],
    y = np_train[1],
    batch_size=batch_size,
    shuffle=True,
    epochs=total_epochs,
    callbacks=[best_model_callback],
    validation_data=np_val,
    class_weight=class_weight_dict
)

linear_eval_best_model = tf.keras.models.load_model(linear_eval_best_model_file_name)

print("Model with lowest validation Loss:")
print(simclr_utitlities.evaluate_model_simple(linear_eval_best_model.predict(np_test[0]), np_test[1], return_dict=True))
print("Model in last epoch")
print(simclr_utitlities.evaluate_model_simple(linear_evaluation_model.predict(np_test[0]), np_test[1], return_dict=True))

### Full HAR Model

In [ ]:
total_epochs = 50
batch_size = 200
tag = "full_eval"

simclr_model = tf.keras.models.load_model(simclr_model_save_path)
full_evaluation_model = simclr_models.create_full_classification_model_from_base_model(simclr_model, output_shape, model_name="TPN", intermediate_layer=7, last_freeze_layer=4)

full_eval_best_model_file_name = f"{working_directory}{start_time_str}_simclr_{tag}.keras"
best_model_callback = tf.keras.callbacks.ModelCheckpoint(full_eval_best_model_file_name,
    monitor='val_loss', mode='min', save_best_only=True, save_weights_only=False, verbose=0
)

training_history = full_evaluation_model.fit(
    x = np_train[0],
    y = np_train[1],
    batch_size=batch_size,
    shuffle=True,
    epochs=total_epochs,
    callbacks=[best_model_callback],
    validation_data=np_val,
    class_weight=class_weight_dict
)

full_eval_best_model = tf.keras.models.load_model(full_eval_best_model_file_name)

print("Model with lowest validation Loss:")
print(simclr_utitlities.evaluate_model_simple(full_eval_best_model.predict(np_test[0]), np_test[1], return_dict=True))
print("Model in last epoch")
print(simclr_utitlities.evaluate_model_simple(full_evaluation_model.predict(np_test[0]), np_test[1], return_dict=True))

## Extra: t-SNE Plots

### Parameters

In [ ]:
# Select a model from which the intermediate representations are extracted
target_model = simclr_model 
perplexity = 30.0

### t-SNE Representations

In [ ]:
intermediate_model = simclr_models.extract_intermediate_model_from_base_model(target_model, intermediate_layer=7)
intermediate_model.summary()

embeddings = intermediate_model.predict(np_test[0], batch_size=600)
tsne_model = sklearn.manifold.TSNE(perplexity=perplexity, verbose=1, random_state=42)
tsne_projections = tsne_model.fit_transform(embeddings)

### Plotting

In [ ]:
categories = np.argmax(np_test[1], axis=1)
plt.figure(figsize=(16, 12))
sns.scatterplot(x=tsne_projections[:,0], y=tsne_projections[:,1], hue=categories, palette='tab10', s=50)
plt.legend(range(len(label_list)), label_list)
plt.show()